# 04 -- Feature engineering (DuckDB SQL)

Aggregate each child table to one row per `SK_ID_CURR` and join into a single
modelling table `data/processed/train_features.parquet`.

Sources:
- `application_train_clean.parquet`, `bureau_clean.parquet`, `previous_application_clean.parquet` (from Stage 2)
- raw CSVs for the large tables (bureau_balance, installments, POS, credit card), cleaned inline

Column prefixes: `inst_`, `bur_`, `prev_`, `pos_`, `cc_`, `app_`.

**Leakage rule:** only use relative offsets available at application time
(negative / <= 0 day and month offsets). This notebook asserts that explicitly.

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    APPLICATION_CLEAN_PATH,
    BUREAU_CLEAN_PATH,
    ID_COL,
    PREVIOUS_APP_CLEAN_PATH,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    TARGET_COL,
    TRAIN_FEATURES_PATH,
)

print("duckdb", duckdb.__version__)

# Paths as DuckDB-friendly posix strings
APP = APPLICATION_CLEAN_PATH.as_posix()
BUR = BUREAU_CLEAN_PATH.as_posix()
PREV = PREVIOUS_APP_CLEAN_PATH.as_posix()
BB = (RAW_DATA_DIR / "bureau_balance.csv").as_posix()
INST = (RAW_DATA_DIR / "installments_payments.csv").as_posix()
POS = (RAW_DATA_DIR / "POS_CASH_balance.csv").as_posix()
CC = (RAW_DATA_DIR / "credit_card_balance.csv").as_posix()

con = duckdb.connect()
con.execute("PRAGMA threads=4")
print("sources ready")

## Leakage guard helper

Every child aggregation filters to offsets available at application time.
We verify the filtered sources contain no positive offsets before aggregating.

In [ ]:
def check_max_offset(source_sql: str, offset_col: str, label: str) -> None:
    q = f"SELECT max({offset_col}) AS mx, min({offset_col}) AS mn FROM ({source_sql})"
    row = con.execute(q).fetchone()
    mx, mn = row
    print(f"{label}: {offset_col} range [{mn}, {mx}]")
    assert mx is None or mx <= 0, f"LEAKAGE: {label} has positive {offset_col} = {mx}"


check_max_offset(f"SELECT DAYS_CREDIT FROM read_parquet('{BUR}') WHERE DAYS_CREDIT <= 0", "DAYS_CREDIT", "bureau")
check_max_offset(f"SELECT MONTHS_BALANCE FROM read_csv_auto('{BB}') WHERE MONTHS_BALANCE <= 0", "MONTHS_BALANCE", "bureau_balance")
check_max_offset(f"SELECT DAYS_DECISION FROM read_parquet('{PREV}') WHERE DAYS_DECISION <= 0", "DAYS_DECISION", "previous_application")
check_max_offset(f"SELECT DAYS_INSTALMENT FROM read_csv_auto('{INST}') WHERE DAYS_INSTALMENT <= 0", "DAYS_INSTALMENT", "installments")
check_max_offset(f"SELECT MONTHS_BALANCE FROM read_csv_auto('{POS}') WHERE MONTHS_BALANCE <= 0", "MONTHS_BALANCE", "POS_CASH")
check_max_offset(f"SELECT MONTHS_BALANCE FROM read_csv_auto('{CC}') WHERE MONTHS_BALANCE <= 0", "MONTHS_BALANCE", "credit_card")
print("\nNo positive offsets in any filtered source.")

## inst_ : installments repayment behaviour

days past due, late rate, underpayment share, payment ratio, and late-rate
trend over last 3 / 6 / 12 months of relative time.

In [ ]:
sql_inst = f"""
WITH inst AS (
    SELECT
        SK_ID_CURR,
        DAYS_INSTALMENT,
        (DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT) AS days_past_due,
        (AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)) AS pay_ratio,
        CASE WHEN AMT_PAYMENT < AMT_INSTALMENT THEN 1 ELSE 0 END AS is_underpay,
        CASE WHEN DAYS_ENTRY_PAYMENT > DAYS_INSTALMENT THEN 1 ELSE 0 END AS is_late
    FROM read_csv_auto('{INST}')
    WHERE DAYS_INSTALMENT <= 0
)
SELECT
    SK_ID_CURR,
    COUNT(*)                                            AS inst_count,
    AVG(days_past_due)                                  AS inst_dpd_mean,
    MAX(days_past_due)                                  AS inst_dpd_max,
    AVG(is_late)                                        AS inst_late_rate,
    AVG(is_underpay)                                    AS inst_underpay_share,
    AVG(pay_ratio)                                      AS inst_pay_ratio_mean,
    AVG(CASE WHEN DAYS_INSTALMENT >= -90  THEN is_late END) AS inst_late_rate_3m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -180 THEN is_late END) AS inst_late_rate_6m,
    AVG(CASE WHEN DAYS_INSTALMENT >= -365 THEN is_late END) AS inst_late_rate_12m
FROM inst
GROUP BY SK_ID_CURR
"""

inst_df = con.execute(sql_inst).df()
print("inst features:", inst_df.shape)
display(inst_df.head())
display(inst_df.describe().T[["mean", "min", "max"]])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(inst_df["inst_late_rate"].dropna(), bins=40, ax=axes[0], color="steelblue")
axes[0].set_title("inst_late_rate")
sns.histplot(inst_df["inst_pay_ratio_mean"].clip(0, 2).dropna(), bins=40, ax=axes[1], color="teal")
axes[1].set_title("inst_pay_ratio_mean (clipped 0-2)")
plt.tight_layout()
plt.show()

## bur_ : bureau credit history

active/closed counts, overdue amounts, utilisation, max days overdue, history length.

In [ ]:
sql_bur = f"""
SELECT
    SK_ID_CURR,
    COUNT(*)                                                       AS bur_count,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Active' THEN 1 ELSE 0 END)       AS bur_active_count,
    SUM(CASE WHEN CREDIT_ACTIVE = 'Closed' THEN 1 ELSE 0 END)       AS bur_closed_count,
    SUM(AMT_CREDIT_SUM_OVERDUE)                                     AS bur_overdue_amt_sum,
    MAX(AMT_CREDIT_MAX_OVERDUE)                                     AS bur_max_overdue_amt,
    MAX(CREDIT_DAY_OVERDUE)                                         AS bur_max_days_overdue,
    SUM(AMT_CREDIT_SUM_DEBT) / NULLIF(SUM(AMT_CREDIT_SUM), 0)       AS bur_utilisation,
    (-MIN(DAYS_CREDIT))                                            AS bur_history_length_days,
    AVG(AMT_CREDIT_SUM)                                             AS bur_amt_credit_sum_mean
FROM read_parquet('{BUR}')
WHERE DAYS_CREDIT <= 0
GROUP BY SK_ID_CURR
"""

bur_df = con.execute(sql_bur).df()
print("bureau features:", bur_df.shape)
display(bur_df.head())
display(bur_df.describe().T[["mean", "min", "max"]])

## bur_bb_ : bureau_balance monthly status (joined via SK_ID_BUREAU)

In [ ]:
sql_bb = f"""
WITH bb AS (
    SELECT
        SK_ID_BUREAU,
        COUNT(*) AS months,
        AVG(CASE WHEN STATUS IN ('1','2','3','4','5') THEN 1 ELSE 0 END) AS dpd_share
    FROM read_csv_auto('{BB}')
    WHERE MONTHS_BALANCE <= 0
    GROUP BY SK_ID_BUREAU
),
map AS (
    SELECT SK_ID_CURR, SK_ID_BUREAU FROM read_parquet('{BUR}')
)
SELECT
    m.SK_ID_CURR,
    AVG(bb.dpd_share)  AS bur_bb_dpd_share_mean,
    SUM(bb.months)     AS bur_bb_months_total
FROM map m
JOIN bb USING (SK_ID_BUREAU)
GROUP BY m.SK_ID_CURR
"""

bb_df = con.execute(sql_bb).df()
print("bureau_balance features:", bb_df.shape)
display(bb_df.head())
display(bb_df.describe().T[["mean", "min", "max"]])

## prev_ : previous applications

approval / refusal rates, time since last application, requested vs approved amounts.

In [ ]:
sql_prev = f"""
SELECT
    SK_ID_CURR,
    COUNT(*)                                                          AS prev_count,
    AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Approved' THEN 1 ELSE 0 END) AS prev_approved_rate,
    AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused'  THEN 1 ELSE 0 END) AS prev_refused_rate,
    (-MAX(DAYS_DECISION))                                             AS prev_days_since_last,
    AVG(AMT_APPLICATION)                                              AS prev_amt_application_mean,
    AVG(AMT_CREDIT)                                                   AS prev_amt_credit_mean,
    AVG(AMT_CREDIT / NULLIF(AMT_APPLICATION, 0))                      AS prev_credit_to_application_mean
FROM read_parquet('{PREV}')
WHERE DAYS_DECISION <= 0
GROUP BY SK_ID_CURR
"""

prev_df = con.execute(sql_prev).df()
print("previous_application features:", prev_df.shape)
display(prev_df.head())
display(prev_df.describe().T[["mean", "min", "max"]])

fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(prev_df["prev_approved_rate"].dropna(), bins=30, ax=ax, color="steelblue")
ax.set_title("prev_approved_rate")
plt.tight_layout()
plt.show()

## pos_ : POS cash balance delinquency + trend

In [ ]:
sql_pos = f"""
WITH pos AS (
    SELECT * FROM read_csv_auto('{POS}') WHERE MONTHS_BALANCE <= 0
)
SELECT
    SK_ID_CURR,
    COUNT(*)                                             AS pos_months_count,
    AVG(SK_DPD)                                          AS pos_dpd_mean,
    MAX(SK_DPD)                                          AS pos_dpd_max,
    MAX(SK_DPD_DEF)                                      AS pos_dpd_def_max,
    SUM(CASE WHEN SK_DPD > 0 THEN 1 ELSE 0 END)          AS pos_delinquency_count,
    (arg_max(CNT_INSTALMENT_FUTURE, MONTHS_BALANCE)
        - arg_min(CNT_INSTALMENT_FUTURE, MONTHS_BALANCE)) AS pos_future_inst_trend
FROM pos
GROUP BY SK_ID_CURR
"""

pos_df = con.execute(sql_pos).df()
print("POS features:", pos_df.shape)
display(pos_df.head())
display(pos_df.describe().T[["mean", "min", "max"]])

## cc_ : credit card utilisation + trend

In [ ]:
sql_cc = f"""
WITH cc AS (
    SELECT
        *,
        AMT_BALANCE / NULLIF(AMT_CREDIT_LIMIT_ACTUAL, 0) AS util
    FROM read_csv_auto('{CC}')
    WHERE MONTHS_BALANCE <= 0
)
SELECT
    SK_ID_CURR,
    COUNT(*)                                             AS cc_months_count,
    AVG(util)                                            AS cc_utilisation_mean,
    MAX(util)                                            AS cc_utilisation_max,
    MAX(SK_DPD)                                          AS cc_dpd_max,
    AVG(AMT_DRAWINGS_CURRENT)                            AS cc_drawings_mean,
    (arg_max(AMT_BALANCE, MONTHS_BALANCE)
        - arg_min(AMT_BALANCE, MONTHS_BALANCE))          AS cc_balance_trend
FROM cc
GROUP BY SK_ID_CURR
"""

cc_df = con.execute(sql_cc).df()
print("credit_card features:", cc_df.shape)
display(cc_df.head())
display(cc_df.describe().T[["mean", "min", "max"]])

## app_ : application-level ratios and base fields

Ratios were derived in Stage 2 cleaning; pass through plus age / employment and
external scores for modelling.

In [ ]:
sql_app = f"""
SELECT
    SK_ID_CURR,
    TARGET,
    app_credit_to_income,
    app_annuity_to_income,
    app_credit_to_goods,
    AGE_YEARS                 AS app_age_years,
    YEARS_EMPLOYED            AS app_years_employed,
    AMT_INCOME_TOTAL          AS app_amt_income_total,
    AMT_CREDIT                AS app_amt_credit,
    AMT_ANNUITY               AS app_amt_annuity,
    EXT_SOURCE_1              AS app_ext_source_1,
    EXT_SOURCE_2              AS app_ext_source_2,
    EXT_SOURCE_3              AS app_ext_source_3,
    CNT_CHILDREN              AS app_cnt_children,
    CNT_FAM_MEMBERS           AS app_cnt_fam_members
FROM read_parquet('{APP}')
"""

app_df = con.execute(sql_app).df()
print("application base:", app_df.shape)
print("target rate:", round(app_df[TARGET_COL].mean(), 4))
display(app_df.head())

## Join into the modelling table

Left join every child aggregate onto application. Missing (no child record)
stays NaN; counts are filled with 0 since 'no history' is informative.

In [ ]:
con.register("app_df", app_df)
con.register("inst_df", inst_df)
con.register("bur_df", bur_df)
con.register("bb_df", bb_df)
con.register("prev_df", prev_df)
con.register("pos_df", pos_df)
con.register("cc_df", cc_df)

sql_join = """
SELECT *
FROM app_df a
LEFT JOIN inst_df USING (SK_ID_CURR)
LEFT JOIN bur_df  USING (SK_ID_CURR)
LEFT JOIN bb_df   USING (SK_ID_CURR)
LEFT JOIN prev_df USING (SK_ID_CURR)
LEFT JOIN pos_df  USING (SK_ID_CURR)
LEFT JOIN cc_df   USING (SK_ID_CURR)
"""
features = con.execute(sql_join).df()
print("joined features:", features.shape)

# Count-style columns: NaN means no history -> 0
count_cols = [c for c in features.columns if c.endswith(("_count", "_months_total", "_months_count", "_delinquency_count"))]
features[count_cols] = features[count_cols].fillna(0)
print("count-like columns filled with 0:", count_cols)

In [ ]:
# Integrity checks
assert features[ID_COL].is_unique, "SK_ID_CURR not unique in feature table"
assert features[ID_COL].notna().all()
assert len(features) == len(app_df), "row count changed vs application base"
print("rows:", len(features), "cols:", features.shape[1])

prefixes = ("inst_", "bur_", "prev_", "pos_", "cc_", "app_")
feat_cols = [c for c in features.columns if c.startswith(prefixes)]
print("prefixed feature columns:", len(feat_cols))

miss = (features[feat_cols].isna().mean() * 100).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 10))
sns.barplot(x=miss.values, y=miss.index, ax=ax, color="salmon")
ax.set_xlabel("% missing")
ax.set_title("Feature missingness (left-join gaps = no history)")
plt.tight_layout()
plt.show()

In [ ]:
# Quick signal check: univariate correlation of features with TARGET
num = features[feat_cols].select_dtypes("number")
corr = num.corrwith(features[TARGET_COL]).dropna().sort_values()
top = pd.concat([corr.head(10), corr.tail(10)])
fig, ax = plt.subplots(figsize=(8, 7))
sns.barplot(x=top.values, y=top.index, ax=ax, color="teal")
ax.set_title("Feature correlation with TARGET (top +/-)")
ax.set_xlabel("pearson r")
plt.tight_layout()
plt.show()
display(top.to_frame("corr_with_target"))

In [ ]:
features.to_parquet(TRAIN_FEATURES_PATH, index=False)
print("wrote", TRAIN_FEATURES_PATH)
print("shape", features.shape)
con.close()

## Summary

- One row per `SK_ID_CURR`, prefixed features: `inst_`, `bur_`, `prev_`, `pos_`, `cc_`, `app_`.
- Leakage guard: all child aggregations filter to `<= 0` relative offsets; asserted above.
- Output: `data/processed/train_features.parquet`.

Next: port each SQL block into `sql/*.sql` and `src/build_features.py` (runnable via `make features`), then add a leakage unit test that scans the SQL for forbidden positive-offset filters.